# Set up and run the framework
Use this notebook locally or on a remote GPU; edit the settings once, run the selected methods, and run the plotting cells when you want figures.

Prepare the project and its imports; on Google Colab this also downloads the latest committed code and installs notebook dependencies.

In [2]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"

if ON_COLAB:
    PROJECT_ROOT = Path("/content/FLFrameWork")
    if (PROJECT_ROOT / ".git").exists():
        if "configuration" in sys.modules or "main" in sys.modules:
            raise RuntimeError("Restart the Python session before fetching code so old functions are not kept in memory.")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "JPNotebooks" / "requirements.txt")], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "JPNotebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "configuration.py").is_file():
    raise FileNotFoundError("Open this notebook from the project folder, or set PROJECT_ROOT to your checkout.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from configuration import configuration
from support_modules.experiment import prepare_runs, run_experiments, comparison_csvs

print("Project:", PROJECT_ROOT)
print("Device:", configuration().device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project: /content/FLFrameWork
Device: cuda
GPU: NVIDIA L4


Results are saved as Results/<dataset>/RUN1, RUN2, RUN3/<selfish|non_selfish>/<method>.csv (one RUN per seed); on Google Colab the same files are copied to Drive after every round.

In [3]:
import hashlib
import shutil
import uuid
import support_modules.experiment as experiment

RESULTS_ROOT = (PROJECT_ROOT / "Results").resolve()
BACKUP_ROOT = None
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive/FLFrameWork_S1_A/Results")
    BACKUP_ROOT.mkdir(parents=True, exist_ok=True)


def safe_backup_path(folder, backup_root):
    """Map Results/<dataset>/RUNn/<scenario> to Drive, preserving each folder."""
    if backup_root is None:
        return None
    relative = Path(folder).resolve().relative_to(RESULTS_ROOT)
    return Path(backup_root) / relative


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_backup_csv(path, backup_folder):
    """Copy ONE output (CSV/PDF) to Drive, then hash-verify; raise on failure."""
    if backup_folder is None:
        if ON_COLAB:
            raise RuntimeError("Google Drive backup is not configured; refusing unprotected Colab training")
        return  # A local Jupyter run has no configured Google Drive mount.

    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Expected experiment result is missing: {source}")

    destination_dir = Path(backup_folder)
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / source.name
    if source.resolve() == destination.resolve():
        return

    # Temporary copy lives beside final file; replace affects ONLY this CSV/PDF.
    partial = destination_dir / f".{source.name}.{uuid.uuid4().hex}.partial"
    try:
        shutil.copy2(source, partial)
        expected = _sha256(source)
        if _sha256(partial) != expected:
            raise IOError(f"Drive upload did not match local output: {destination}")
        os.replace(partial, destination)
        if _sha256(destination) != expected:
            raise IOError(f"Drive backup verification failed: {destination}")
    finally:
        partial.unlink(missing_ok=True)
    print("[DRIVE VERIFIED]", destination)

# run_experiments() resolves both helpers from its module's global namespace.
experiment.backup_path = safe_backup_path
experiment.backup_csv = safe_backup_csv
backup_path = safe_backup_path
backup_csv = safe_backup_csv

if ON_COLAB:
    probe = RESULTS_ROOT / f"_drive_probe_{uuid.uuid4().hex}.txt"
    probe.write_text("Google Drive backup write/read verification", encoding="utf-8")
    target = BACKUP_ROOT / probe.name
    try:
        backup_csv(probe, BACKUP_ROOT)
        if target.read_bytes() != probe.read_bytes():
            raise IOError("Google Drive probe mismatch")
    finally:
        probe.unlink(missing_ok=True)
        target.unlink(missing_ok=True)
    print("\nGOOGLE DRIVE BACKUP VERIFIED:", BACKUP_ROOT)
    print("Training CSVs are backed up after EACH fully evaluated round.")
else:
    print("Local environment: results are local (no Colab Drive mount).")

Mounted at /content/drive
[DRIVE VERIFIED] /content/drive/MyDrive/FLFrameWork_S1_A/Results/_drive_probe_3f5fd70663b84d6f875565a46d3fb02a.txt

GOOGLE DRIVE BACKUP VERIFIED: /content/drive/MyDrive/FLFrameWork_S1_A/Results
Training CSVs are backed up after EACH fully evaluated round.


Choose datasets, methods and seeds here; the five-round smoke run uses one seed to check the setup before the full three-seed experiment.

In [5]:
config = configuration()

DATASETS = ["tinyimagenet"]
# METHODS = ["fedavg", "fairrfl", "fedcda", "fedmut", "fcfl"]
METHODS = ["fedavg", "fairrfl"]
SEEDS = [42, 43, 44]
SMOKE_SEEDS = [42]
SMOKE_TEST = False
RERUN_COMPLETED = False

# None runs all seeds; [43] runs only seed 43 while keeping its output in RUN2.
ONLY_SEEDS = [42]

config.num_of_clients = 30
config.rounds = 5 if SMOKE_TEST else 40
config.local_epochs = 5
config.batch_size = 64
config.lr = 0.01
config.dirichlet_alpha = 0.5
MOMENTUM = {"mnist": 0.5, "cifar10": 0.9, "tinyimagenet": 0.9}

# False = the honest, non_selfish runs used in the setup's Table II.
config.selfishness = False
config.selfish_fraction = 0.05
config.selfish_strength = 0.7
config.selfish_start_round = 3

config.fairrfl_q = 1.0
config.fairrfl_tau = 2.5
config.fairrfl_dynamic_q = True
config.fairrfl_lipschitz = None
config.fedcda_warmup = 1 if SMOKE_TEST else 10
config.fcfl_alpha = 0.5
config.fcfl_fraction = 1.0

if ONLY_SEEDS is not None and not set(ONLY_SEEDS).issubset(set(SEEDS)):
    raise ValueError(f"ONLY_SEEDS must contain only the seeds in SEEDS={SEEDS}")

runs = prepare_runs(
    config, DATASETS, METHODS, RESULTS_ROOT, MOMENTUM,
    SMOKE_SEEDS if SMOKE_TEST else SEEDS, SMOKE_TEST
)
runs = [run for run in runs if ONLY_SEEDS is None or run[0].seed in ONLY_SEEDS]
if not runs:
    raise RuntimeError("No runs selected; check ONLY_SEEDS and SMOKE_TEST")

print("Results root (local):", RESULTS_ROOT)
print("Google Drive root:", BACKUP_ROOT)
for run_config, folder in runs:
    rel = (folder / (run_config.method + ".csv")).relative_to(RESULTS_ROOT)
    print(f"Seed {run_config.seed}: {rel}")
    if BACKUP_ROOT is not None:
        print("  -> Drive:", BACKUP_ROOT / rel)
print("Rounds:", config.rounds, "| Epochs:", config.local_epochs,
      "| Selected seeds:", sorted({r[0].seed for r in runs}))
print("Selfish client IDs (0-indexed):", runs[0][0].selfish_client_ids)

Results root (local): /content/FLFrameWork/Results
Google Drive root: /content/drive/MyDrive/FLFrameWork_S1_A/Results
Seed 42: tinyimagenet/RUN1/non_selfish/fedavg.csv
  -> Drive: /content/drive/MyDrive/FLFrameWork_S1_A/Results/tinyimagenet/RUN1/non_selfish/fedavg.csv
Seed 42: tinyimagenet/RUN1/non_selfish/fairrfl.csv
  -> Drive: /content/drive/MyDrive/FLFrameWork_S1_A/Results/tinyimagenet/RUN1/non_selfish/fairrfl.csv
Rounds: 40 | Epochs: 5 | Selected seeds: [42]
Selfish client IDs (0-indexed): []


Run the selected methods in order; completed runs are skipped, a method that diverges (NaN) is recorded and skipped, and incomplete runs restart from round 1 because CSVs are not model checkpoints.

In [ ]:
if ON_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime before training; loading existing results needs no GPU.")
if ON_COLAB and BACKUP_ROOT is None:
    raise RuntimeError("Drive must be mounted and verified before training")

run_summary = run_experiments(
    runs, backup_root=BACKUP_ROOT, rerun_completed=RERUN_COMPLETED
)
display(run_summary)

100%|██████████| 248M/248M [02:28<00:00, 1.67MB/s]


Device: cuda
Dataset: tinyimagenet
Method: fedavg
Clients: 30
Rounds: 40
Training batch size: 64
Selfish clients: []
Local epochs: 5
Training samples: 100000
Testing samples: 10000
Client 0 training samples: 3685 testing samples: 374
Client 1 training samples: 3112 testing samples: 313
Client 2 training samples: 3031 testing samples: 308
Client 3 training samples: 3314 testing samples: 340
Client 4 training samples: 3688 testing samples: 361
Client 5 training samples: 3180 testing samples: 322
Client 6 training samples: 3402 testing samples: 343
Client 7 training samples: 2944 testing samples: 296
Client 8 training samples: 3271 testing samples: 327
Client 9 training samples: 3550 testing samples: 355
Client 10 training samples: 3017 testing samples: 297
Client 11 training samples: 3297 testing samples: 333
Client 12 training samples: 3367 testing samples: 342
Client 13 training samples: 3171 testing samples: 312
Client 14 training samples: 3786 testing samples: 384
Client 15 training 

In [ ]:

from google.colab import runtime

print("✅ All tasks completed! Disconnecting runtime to save credits...")

runtime.unassign()


Optional diagnostic: inspect accuracy over rounds and FairRFL update sizes, including stopped runs, and save this check under temp rather than among final figures.

In [ ]:
from analysis.plots import plot_run_diagnostics

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    diagnostic_path = PROJECT_ROOT / "temp" / "diagnostics" / folder.relative_to(RESULTS_ROOT) / "training_diagnostic.pdf"
    plot_run_diagnostics(csv_data, diagnostic_path)
    print("Diagnostic PDF:", diagnostic_path)

Run this cell when you want final client-accuracy plots and fairness summaries; only complete matching runs are accepted, and PDFs are saved beside their CSVs.

In [ ]:
from analysis.plots import final_client_scores, plot_client_test_accuracy
from support_modules.experiment import backup_csv, backup_path

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    scores, settings = final_client_scores(csv_data)
    pdf_path = folder / "final_test_accuracy_all_methods.pdf"
    summary = plot_client_test_accuracy(scores, settings, pdf_path)
    display(summary)
    backup_folder = backup_path(folder, BACKUP_ROOT)
    backup_csv(pdf_path, backup_folder)
    print("Saved PDF:", pdf_path)
    if backup_folder is not None:
        print("Drive folder:", backup_folder)

Run this after every seed has finished to print Table II from the setup: final-round metrics as mean ± sample standard deviation over the seeds, saved as summary_<scenario>.csv in the dataset folder.

In [ ]:
from analysis.results import seed_summary
from analysis.plots import format_seed_summary
from support_modules.experiment import backup_csv, backup_path



# Group the RUN folders of each dataset and scenario; Table II is saved as summary_<scenario>.csv in the dataset folder.
comparisons = {}
for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    comparisons.setdefault((folder.parent.parent, folder.name), {})[folder.parent.name] = csv_data

for (dataset_folder, scenario), csv_by_seed in comparisons.items():
    means, deviations, per_seed, partial = seed_summary(csv_by_seed)
    summary = format_seed_summary(means, deviations)
    summary_path = dataset_folder / f"summary_{scenario}.csv"
    per_seed_path = dataset_folder / f"summary_{scenario}_per_seed.csv"
    summary.to_csv(summary_path)
    per_seed.to_csv(per_seed_path, index=False)
    print(dataset_folder.relative_to(RESULTS_ROOT), scenario, "| RUNs:", sorted(csv_by_seed), "| seeds:", sorted(int(seed) for seed in per_seed["seed"].unique()))
    for method, missing in partial.items():
        print(f"{method} diverged for seeds {missing}; it is left out of the mean ± std table.")
    display(summary)
    backup_folder = backup_path(dataset_folder, BACKUP_ROOT)
    backup_csv(summary_path, backup_folder)
    backup_csv(per_seed_path, backup_folder)
    print("Saved Table II:", summary_path)

Optional: choose one method and generate its final client-accuracy plot in each selected dataset folder.

In [ ]:
from support_modules.experiment import backup_csv, backup_path

METHOD = "fairrfl"

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    scores, settings = final_client_scores(csv_data)
    if METHOD not in scores.columns:
        raise ValueError(f"Choose a loaded method: {list(scores.columns)}")
    pdf_path = folder / f"final_test_accuracy_{METHOD}.pdf"
    display(plot_client_test_accuracy(scores[[METHOD]], settings, pdf_path))
    backup_folder = backup_path(folder, BACKUP_ROOT)
    backup_csv(pdf_path, backup_folder)
    print("Saved PDF:", pdf_path)

Selfish and non-selfish runs use separate selfish/non_selfish folders inside each RUN, and smoke tests use smoke_<dataset>; to rerun with different settings, first move or rename the existing dataset folder; one of 30 clients is selected at fraction 0.05 (0.3 gives nine). Keep the same seed and shared settings for both scenarios. No setting guarantees a particular ranking; report the measured results. Compare honest-client mean, worst accuracy and standard deviation together. After pulling new Python code, restart the kernel and rerun setup before training.